In [6]:
import pandas as pd
from dotenv import load_dotenv
import os
from datetime import date, timedelta
import time
import json
import ijson
from sqlalchemy import create_engine
from tqdm import tqdm

In [7]:
import sys
sys.path.append('./lib')

import library_st_data_processing as lsdp

In [8]:
# load .env from the current directory (or specify a path)
load_dotenv(dotenv_path=".env")
api_key = os.getenv("ST_API_KEY")

In [9]:
base_url = 'https://api.sensortower.com'

In [10]:
today_str = date.today().strftime("%Y-%m-%d")
timestamp = int(time.time())

# Base Layer

**Get Raw File: Top Game Annual Performance**

In [5]:
# Manual Download

**Get Raw File: Game Full Info**

In [8]:
# Load Top Game Annual Performance file
df_top_game_annual_performance = pd.read_csv("data/base/st_webdownload_annual_game_performance_2025-10-23.csv")

In [10]:
# Get the unified_app_ids
unified_app_ids = set(list(df_top_game_annual_performance["Unified ID"]))

In [18]:
# Retrieve Game Full Info file
df_game_full_info = lsdp.retrieve_full_info_game_table(api_key, base_url, unified_app_ids)

Get canonical app data...
making call for chunk number 1
call successful!
making call for chunk number 2
call successful!
making call for chunk number 3
call successful!
making call for chunk number 4
call successful!
making call for chunk number 5
call successful!
making call for chunk number 6
call successful!
making call for chunk number 7
call successful!
making call for chunk number 8
call successful!
making call for chunk number 9
call successful!
making call for chunk number 10
call successful!
making call for chunk number 11
call successful!
making call for chunk number 12
call successful!
making call for chunk number 13
call successful!
making call for chunk number 14
call successful!
making call for chunk number 15
call successful!
making call for chunk number 16
call successful!
making call for chunk number 17
call successful!
making call for chunk number 18
call successful!
making call for chunk number 19
call successful!
making call for chunk number 20
call successful!
mak

In [19]:
# Export the file to csv and save to base layer folder
df_game_full_info.to_csv('data/base/st_api_game_full_info_{}.csv'.format(timestamp), index=False)

**Get Raw File: App Full Info**

In [6]:
# Read file Game Full Info
df_game_full_info = pd.read_csv('data/base/st_api_game_full_info_2025-10-23.csv')

In [8]:
# Retrieve file App Full Info
df_app_full_info = lsdp.get_local_app_info_from_game_full_info_table(api_key, base_url, df_game_full_info)

Extracting app list from game table...
Getting iOS apps info...
making call for chunk number 1
call successful!
making call for chunk number 2
call successful!
making call for chunk number 3
call successful!
making call for chunk number 4
call successful!
making call for chunk number 5
call successful!
making call for chunk number 6
call successful!
making call for chunk number 7
call successful!
making call for chunk number 8
call successful!
making call for chunk number 9
call successful!
making call for chunk number 10
call successful!
making call for chunk number 11
call successful!
making call for chunk number 12
call successful!
making call for chunk number 13
call successful!
making call for chunk number 14
call successful!
making call for chunk number 15
call successful!
making call for chunk number 16
call successful!
making call for chunk number 17
call successful!
making call for chunk number 18
call successful!
making call for chunk number 19
call successful!
making call fo

In [9]:
# Export the file to csv and save to base layer folder
df_app_full_info.to_csv('data/base/st_api_app_full_info_{}.csv'.format(timestamp), index=False)

**Get Raw File: Mapping Publisher to Apps (Top Down)**

In [7]:
# Manually Prepared

**Get Raw File: Mapping Publisher to Apps and Publisher IDs (Top Down)**

In [20]:
# Path to Mapping Publisher to Apps Raw File
path_mapping_publisher_to_apps = 'data/base/st_manualsearch_mapping_publisher_to_apps_top_down_2025-10-28.csv'

In [21]:
# Retrieve the publisherid from ST
df_mapping_publisher_to_apps_and_publisherids = lsdp.create_mapping_publisher_to_apps_publisherids(path_mapping_publisher_to_apps, api_key, base_url)

Reading csv file: mapping publisher to apps...
Get ST publisher info for ios apps...
making call for chunk number 1
call successful!
making call for chunk number 2
call successful!
making call for chunk number 3
call successful!
making call for chunk number 4
call successful!
Get ST publisher info for Android apps...
making call for chunk number 1
call successful!
making call for chunk number 2
call successful!
making call for chunk number 3
call successful!
making call for chunk number 4
call successful!


In [22]:
# Export the file to csv and save to base layer folder
df_mapping_publisher_to_apps_and_publisherids.to_csv('data/base/st_api_mapping_publisher_to_apps_publisherids_top_down_{}.csv'.format(timestamp), index=False)

**Get Raw File: Mapping Publisher to Publisher ID and Revenue Multiplier (Bottom Up)**

In [12]:
# Manually Prepared

**Get Raw File: Mapping App to Revenue Multiplier (Special Cases)**

In [ ]:
# Manually Prepared

**Get Raw File: App Performance (Daily)**

In [6]:
# Read file App Full Info (Adjusted)
df_app_full_info_adjusted = pd.read_csv('data/staging/st_app_full_info_adjusted_1763107898.csv', low_memory=False)

In [ ]:
# Run the function to fetch the data and export to json
app_performance_data = lsdp.create_table_app_performance_grouped_by_game_daily(
    api_key, 
    base_url, 
    str_start_date = '2014-01-01', 
    str_end_date = (date.today() - timedelta(days=3)).isoformat(), 
    df_app_full_info_adjusted, 
    json_export_path = 'data/base')

# Staging Layer

**Get Staging File: Mapping Publisher to Publisher ID and Revenue Multiplier (Full)**

In [6]:
# Read file Mapping Publisher to Apps and Publisher IDs (Top Down)

data_types = {
    "os_x": "string",
    "cleaned_publisher_name": "string",
    "game_name": "string",
    "sensor_tower_link": "string",
    "app_id_trimmed": "string",
    "publisher_id": "string",
    "publisher_name": "string",
}

df_mapping_publisher_to_apps_and_publisherids_top_down = pd.read_csv(
    'data/base/st_api_mapping_publisher_to_apps_publisherids_top_down_2025-10-28.csv',
    dtype = data_types
)

In [7]:
# Read file Mapping Publisher to Publisher ID and Revenue Multiplier (Bottom Up)

data_types = {
    "cleaned_publisher_name": "string",
    "publisher_id": "string",
    "publisher_name": "string",
    "revenue_multiplier": "int64"
}

df_mapping_publisher_to_publisherid_revenue_multiplier_bottom_up = pd.read_csv(
    'data/base/st_manualsearch_mapping_publisher_to_publisherids_revenue_multiplier_bottom_up_2025-10-31.csv',
    dtype = data_types
)

In [8]:
# Merge the top-down and bottom-up tables into a full mapping

In [9]:
df_mapping_publisher_to_publisherid_revenue_multiplier_full = lsdp.create_mapping_publisher_to_publisherids_revenue_multiplier(
    df_mapping_publisher_to_apps_and_publisherids_top_down,
    df_mapping_publisher_to_publisherid_revenue_multiplier_bottom_up
)

Merging is complete, now have the full mapping table. But need to double check for rows having null publisher_id
These are rows having null publisher_id:
             cleaned_publisher_name publisher_id publisher_name  \
138  CTCP Giai Tri Thien Thuong Hoa         <NA>           <NA>   

     revenue_multiplier  
138                   3  
Best to check again these rows. Potential reasons: apps become inactive in the country of interest; app id typo; app id changed by SensorTower; etc.
Removing these rows from the merged mapping table...


In [10]:
# Export the file to csv and save to staging layer folder
df_mapping_publisher_to_publisherid_revenue_multiplier_full.to_csv("data/staging/st_mapping_publisher_to_publisherids_revenue_multiplier_full_{}.csv".format(timestamp),index=False)

**Get Staging File: App Full Info (Adjusted)**

In [6]:
# Read file App Full Info
df_app_full_info = pd.read_csv('data/base/st_api_app_full_info_2025-10-27.csv', low_memory=False)

In [7]:
# Read file Mapping Publisher to Publisher ID and Revenue Multiplier (Full)
df_mapping_publisher_to_publiserid_and_revenue_multiplier_full = pd.read_csv('data/staging/st_mapping_publisher_to_publisherids_revenue_multiplier_full_2025-10-31.csv', low_memory=False)

In [8]:
# Read file Mapping App to Revenue Multiplier (Special Cases)
df_mapping_app_to_revenue_multiplier_special_case = pd.read_csv('data/base/st_mapping_app_to_revenue_multiplier_special_case_1763095974.csv', low_memory=False)

In [9]:
# Cast cleaned_publisher_name and revenue_multiplier to App Full Info to get App Full Info (Adjusted)
df_app_full_info_adjusted = lsdp.add_cleaned_publisher_name_and_revenue_multiplier_to_app_full_info(
    df_app_full_info,
    df_mapping_publisher_to_publiserid_and_revenue_multiplier_full
)

In [27]:
# Adjust the App Full Info list again for special cases
df_app_full_info_adjusted_again = lsdp.adjust_cleaned_publisher_name_and_revenue_multiplier_of_app_full_info_special_cases(
    df_app_full_info_adjusted,
    df_mapping_app_to_revenue_multiplier_special_case
).drop_duplicates()

In [34]:
# Export the file to csv and save to staging layer folder
df_app_full_info_adjusted_again.to_csv('data/staging/st_app_full_info_adjusted_{}.csv'.format(timestamp), index=False)

**Get Staging File: App Performance (Daily) - Revenue Adjusted [NEED TO EDIT LATER WITH CORRECT FILE PATHS & DO SOME REFACTOR]**

In [22]:
# Read file App Full Info (Adjusted)
df_app_full_info_adjusted = pd.read_csv('data/staging/st_app_full_info_adjusted_1763107898.csv', low_memory=False)

In [14]:
# Specify the path of input and output
src_path = "data/staging/st_app_performance_daily_1764146538.json"
file_timestamp = os.path.basename(src_path).split("_")[-1].split(".")[0]
dst_path = "data/staging/st_app_performance_daily_{}_revenue_adjusted.json".format(file_timestamp)

In [ ]:
# Run the function for adjusting and streaming
lsdp.stream_and_adjust_app_performance_daily_json_file(
    src_path,
    dst_path,
    df_app_full_info_adjusted
)

# Modeling Layer

In [9]:
# read data from json
with open('data/draft/app_performance_data_1764146538.json', 'r') as file:
    app_performance_data = json.load(file)

In [20]:
app_performance_data

[{'aid': 336834650, 'cc': 'VN', 'd': '2014-01-13T00:00:00Z', 'ir': 8},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-17T00:00:00Z', 'ir': 4},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-19T00:00:00Z', 'ir': 6},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-22T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-28T00:00:00Z', 'ir': 1},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-01-29T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-02-13T00:00:00Z', 'ir': 2},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-03-01T00:00:00Z', 'ir': 1},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-03-12T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-06-13T00:00:00Z', 'ir': 2},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-06-19T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-06-22T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-06-27T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-07-04T00:00:00Z'},
 {'aid': 336834650, 'cc': 'VN', 'd': '2014-07-06T00:00:00Z', 'ir': 4}

In [26]:
# group the data by app id and unified id

df_perf = pd.DataFrame(app_performance_data)

In [29]:
df_perf

,aid,cc,d,ir,au,iu,ar,c,u,r,aid_str
0,336834650,VN,2014-01-13T00:00:00Z,8,NaN,NaN,NaN,NaN,NaN,NaN,336834650
1,336834650,VN,2014-01-17T00:00:00Z,4,NaN,NaN,NaN,NaN,NaN,NaN,336834650
2,336834650,VN,2014-01-19T00:00:00Z,6,NaN,NaN,NaN,NaN,NaN,NaN,336834650
3,336834650,VN,2014-01-22T00:00:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,336834650
4,336834650,VN,2014-01-28T00:00:00Z,1,NaN,NaN,NaN,NaN,NaN,NaN,336834650
...,...,...,...,...,...,...,...,...,...,...,...
52747274,world.playme.x,NaN,2025-10-20T00:00:00Z,NaN,NaN,NaN,NaN,VN,NaN,68,world.playme.x
52747275,world.playme.x,NaN,2025-10-21T00:00:00Z,NaN,NaN,NaN,NaN,VN,NaN,81,world.playme.x
52747276,world.playme.x,NaN,2025-10-22T00:00:00Z,NaN,NaN,NaN,NaN,VN,NaN,NaN,world.playme.x
52747277,world.playme.x,NaN,2025-10-23T00:00:00Z,NaN,NaN,NaN,NaN,VN,NaN,NaN,world.playme.x


In [28]:
# Make sure app ids are consistent types (strings in this example)
df_perf["aid_str"] = df_perf["aid"].astype(str)

In [30]:
df_perf["d"] = pd.to_datetime(df_perf["d"])

In [31]:
# Make a string version of app_id too
df_app_full_info["app_id_str"] = df_app_full_info["app_id"].astype(str)

In [34]:
# Convert app id list to strings
list_app_ids_ios_str = [str(a) for a in list_app_ids_ios]
list_app_ids_android_str = [str(a) for a in list_app_ids_android]

In [36]:
list_app_ids_str = list_app_ids_ios_str + list_app_ids_android

In [35]:
# Keep only rows for those app IDs
df_app_ios_info = df_app_full_info[
    df_app_full_info["app_id_str"].isin(list_app_ids_ios_str)
]
df_app_android_info = df_app_full_info[
    df_app_full_info["app_id_str"].isin(list_app_ids_android_str)
]

In [39]:
# Sort by app and date (equivalent to your per-app sorted(..., key=lambda x["d"]))
df_perf = df_perf.sort_values(["aid_str", "d"])

In [ ]:
# For each app, get a list of dicts (one dict per performance row)
perf_lists = (
    df_perf
    .groupby("aid_str", sort=False)
    .apply(lambda g: g.to_dict("records"))
)